In [1]:
"""
Build assumption_master sheet: ci_reinstatement
-----
Extracts CI reinstatement tables and CI-specific scalar loadings.
Consumed by CI pricing rows 27 (reinstatement proportion),
38 (standalone loading), 40 (industry loading), 57 (reinstatement premium).

Schema: feature | key | value | notes
"""

from pathlib import Path
from datetime import datetime, timezone
import hashlib, csv, openpyxl, polars as pl

BASE = Path(r"J:\Product & Client Solutions\Toolbox\Calculators"
            r"\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE_XLSX = BASE / "Basis and Assumptions.xlsx"
SOURCE_SHEET = "Assumptions"

ASSUMPTION_VERSION = "A2026_07"
SNAPSHOT_DIR = BASE / "assumption_snapshots" / f"assumption_{ASSUMPTION_VERSION}"
MASTER_XLSX = BASE / "assumption_master" / f"assumption_master_{ASSUMPTION_VERSION}.xlsx"
BUILD_LOG = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME = "ci_reinstatement"
PARQUET_PATH = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"

BUILT_AT = datetime.now(timezone.utc).isoformat(timespec="seconds")
BUILT_BY = "Mohamed Hussein Gadiwala"

wb = openpyxl.load_workbook(SOURCE_XLSX, data_only=True)
ws = wb[SOURCE_SHEET]
SOURCE_SHA = hashlib.sha256(SOURCE_XLSX.read_bytes()).hexdigest()

print(f"Building: {SHEET_NAME}")

Building: ci_reinstatement


In [2]:
def find_all_cells_any_col(ws, label, max_row=1200, max_col=30):
    hits = []
    for row in range(1, max_row + 1):
        for col in range(1, max_col + 1):
            v = ws.cell(row=row, column=col).value
            if isinstance(v, str) and v.strip() == label:
                hits.append((row, col))
    return hits

def get_col_letter(col):
    return openpyxl.utils.get_column_letter(col)

DIAGNOSTIC_LABELS = [
    "Reinstatement Period",
    "Critical Inc Reinstatement  (accelerator)",   # double space in source
    "Critical Inc Reinstatement Discount Reduction Factors",
    "% of difference between rates",
    "Standalone",
    "Voluntary Loading:",
    "Industry Adjustment",
    "Industry Selected",
    "Loading",
    "Loadings:",
]

print("=" * 90)
print("DIAGNOSTIC — CI reinstatement anchors")
print("=" * 90)

for label in DIAGNOSTIC_LABELS:
    hits = find_all_cells_any_col(ws, label)
    ci_hits = [(r, c) for r, c in hits if 313 <= r <= 342]
    other_hits = [(r, c) for r, c in hits if not (313 <= r <= 342)]
    print(f"\n'{label}': {len(ci_hits)} match(es) in CI block")
    for i, (r, c) in enumerate(ci_hits):
        vals = []
        for dc in range(0, 4):
            v = ws.cell(row=r, column=c + dc).value
            vals.append(f"{get_col_letter(c + dc)}{r}={v!r}")
        print(f"  [{i}] {' | '.join(vals)}")
    if other_hits:
        print(f"  (also {len(other_hits)} match(es) outside CI block, first: row {other_hits[0][0]})")

# Also dump the CI reinstatement area rows 325-345 for context
print("\n" + "=" * 90)
print("CI reinstatement area preview (rows 325-345, cols B-H):")
print("=" * 90)
for row in range(325, 346):
    vals = []
    for col in range(2, 9):  # B-H
        v = ws.cell(row=row, column=col).value
        vals.append(f"{get_col_letter(col)}={v!r}")
    print(f"  Row {row:3d}: {' | '.join(vals)}")

DIAGNOSTIC — CI reinstatement anchors

'Reinstatement Period': 1 match(es) in CI block
  [0] B325='Reinstatement Period' | C325=None | D325=None | E325='Table Mappings'

'Critical Inc Reinstatement  (accelerator)': 1 match(es) in CI block
  [0] B334='Critical Inc Reinstatement  (accelerator)' | C334='% of difference between rates' | D334=None | E334='CI CO'

'Critical Inc Reinstatement Discount Reduction Factors': 1 match(es) in CI block
  [0] E337='Critical Inc Reinstatement Discount Reduction Factors' | F337='Factor' | G337=None | H337=None

'% of difference between rates': 1 match(es) in CI block
  [0] C334='% of difference between rates' | D334=None | E334='CI CO' | F334=None

'Standalone': 1 match(es) in CI block
  [0] G325='Standalone' | H325='Age Definition' | I325='Type' | J325=None

'Voluntary Loading:': 0 match(es) in CI block

'Industry Adjustment': 0 match(es) in CI block

'Industry Selected': 0 match(es) in CI block

'Loading': 0 match(es) in CI block

'Loadings:': 0 match

In [3]:
all_rows: list[dict] = []

def add_row(feature, key, value, source_row, notes=""):
    all_rows.append({
        "feature": feature,
        "key": key,
        "value": value,
        "source_row": source_row,
        "notes": notes,
    })

# =============================================================================
# 1. Reinstatement Period lookup (rows 326-331, cols B/C)
#    Text label → number of days
# =============================================================================
REINST_PERIOD_ROWS = [
    (326, "30_days"),
    (327, "60_days"),
    (328, "90_days"),
    (329, "180_days"),
    (330, "12_months"),
    (331, "24_months"),
]
for row, key in REINST_PERIOD_ROWS:
    source_label = ws.cell(row=row, column=2).value  # B
    days = ws.cell(row=row, column=3).value          # C
    add_row("reinstatement_period", key, days, row,
            f"Period text {source_label!r} → {days} days")

# =============================================================================
# 2. Reinstatement Proportion — Accelerator only (rows 335-340, cols B/C)
#    Applied to (Standalone rate - Accelerator rate) when reinstatement selected.
# =============================================================================
REINST_PROP_ROWS = [
    (335, "30_days"),
    (336, "60_days"),
    (337, "90_days"),
    (338, "180_days"),
    (339, "12_months"),
    (340, "24_months"),
]
for row, key in REINST_PROP_ROWS:
    source_label = ws.cell(row=row, column=2).value
    prop = ws.cell(row=row, column=3).value
    add_row("reinstatement_proportion", key, prop, row,
            f"Reinstatement proportion for {source_label!r} "
            f"(% of difference between standalone and accelerator rates)")

# =============================================================================
# 3. Discount Reduction Factors (rows 338-341, cols E/F)
#    NOTE: source uses "365 days" here for 12 months (label drift)
# =============================================================================
DISCOUNT_ROWS = [
    (338, "30_days"),
    (339, "90_days"),
    (340, "180_days"),
    (341, "365_days"),   # source label; equivalent to "12 months" in other tables
]
for row, key in DISCOUNT_ROWS:
    source_label = ws.cell(row=row, column=5).value  # E
    factor = ws.cell(row=row, column=6).value        # F
    add_row("discount_reduction_factor", key, factor, row,
            f"Discount reduction factor for {source_label!r}. "
            f"Note: source uses '365 days' here whereas period/proportion tables "
            f"use '12 months' for equivalent duration.")

# =============================================================================
# 4. Product exclusion (row 336, cols G/H) — scalar
#    Which CI product option is excluded from reinstatement logic.
# =============================================================================
excluded_product = ws.cell(row=336, column=8).value  # H336
add_row("product_exclusion", "excluded_option", excluded_product, 336,
        f"CI product option excluded from reinstatement logic")

# --- Summary ---
from collections import Counter
feature_counts = Counter(r["feature"] for r in all_rows)
print(f"Total rows harvested: {len(all_rows)}")
print(f"\nRows per feature:")
for feat, cnt in sorted(feature_counts.items()):
    print(f"  {feat:28s}: {cnt}")

print(f"\nAll rows preview:")
for r in all_rows:
    print(f"  {r['feature']:26s} | {r['key']:18s} = {r['value']!r} "
          f"({type(r['value']).__name__})")

Total rows harvested: 17

Rows per feature:
  discount_reduction_factor   : 4
  product_exclusion           : 1
  reinstatement_period        : 6
  reinstatement_proportion    : 6

All rows preview:
  reinstatement_period       | 30_days            = 30 (int)
  reinstatement_period       | 60_days            = 60 (int)
  reinstatement_period       | 90_days            = 90 (int)
  reinstatement_period       | 180_days           = 180 (int)
  reinstatement_period       | 12_months          = 365 (int)
  reinstatement_period       | 24_months          = 730 (int)
  reinstatement_proportion   | 30_days            = 0.958904109589041 (float)
  reinstatement_proportion   | 60_days            = 0.9178082191780822 (float)
  reinstatement_proportion   | 90_days            = 0.8767123287671232 (float)
  reinstatement_proportion   | 180_days           = 0.7534246575342466 (float)
  reinstatement_proportion   | 12_months          = 0.5 (float)
  reinstatement_proportion   | 24_months          = 0

In [4]:
def classify(v) -> str:
    if v is None:            return "null"
    if isinstance(v, bool):  return "bool"
    if isinstance(v, int):   return "int"
    if isinstance(v, float): return "float"
    return "string"

for r in all_rows:
    v = r["value"]
    r["param_type"] = classify(v)
    r["value_str"] = "" if v is None else str(v)

df = pl.DataFrame(all_rows).select([
    "feature",
    "key",
    pl.col("value_str").alias("value"),
    "param_type",
    "source_row",
    "notes",
])

# =============================================================================
# Structural validation
# =============================================================================
key_cols = ["feature", "key"]
dupes = df.group_by(key_cols).len().filter(pl.col("len") > 1)
assert dupes.height == 0, f"Duplicate keys:\n{dupes}"

expected_features = {
    "reinstatement_period",
    "reinstatement_proportion",
    "discount_reduction_factor",
    "product_exclusion",
}
actual_features = set(df["feature"].unique().to_list())
assert actual_features == expected_features, (
    f"Feature mismatch: expected {expected_features}, got {actual_features}"
)

# Row-count expectations
expected_counts = {
    "reinstatement_period":       6,
    "reinstatement_proportion":   6,
    "discount_reduction_factor":  4,
    "product_exclusion":          1,
}
for feat, expected in expected_counts.items():
    actual = df.filter(pl.col("feature") == feat).height
    assert actual == expected, (
        f"Feature {feat}: got {actual} rows, expected {expected}"
    )

# =============================================================================
# Frozen value expectations for A2026_07
# =============================================================================
def get_val(feature, key):
    row = df.filter(
        (pl.col("feature") == feature) & (pl.col("key") == key)
    )
    assert row.height == 1, f"Missing: {feature}/{key}"
    return row["value"][0], row["param_type"][0]

# reinstatement_period (int days)
period_frozen = [
    ("30_days",    30),
    ("60_days",    60),
    ("90_days",    90),
    ("180_days",  180),
    ("12_months", 365),
    ("24_months", 730),
]
for key, expected in period_frozen:
    val, typ = get_val("reinstatement_period", key)
    assert typ == "int", f"period {key} expected int, got {typ}"
    assert int(val) == expected, f"period {key}: got {val}, expected {expected}"

# reinstatement_proportion (float, with high precision)
prop_frozen = [
    ("30_days",    0.958904109589041),
    ("60_days",    0.9178082191780822),
    ("90_days",    0.8767123287671232),
    ("180_days",   0.7534246575342466),
    ("12_months",  0.5),
    ("24_months",  0.0),
]
for key, expected in prop_frozen:
    val, typ = get_val("reinstatement_proportion", key)
    assert typ in ("float", "int"), f"proportion {key}: unexpected type {typ}"
    assert abs(float(val) - expected) < 1e-9, (
        f"proportion {key}: got {val}, expected {expected}"
    )

# discount_reduction_factor
discount_frozen = [
    ("30_days",   0.0),
    ("90_days",   0.125),
    ("180_days",  0.25),
    ("365_days",  0.5),
]
for key, expected in discount_frozen:
    val, typ = get_val("discount_reduction_factor", key)
    assert typ in ("float", "int"), f"discount {key}: unexpected type {typ}"
    assert abs(float(val) - expected) < 1e-9, (
        f"discount {key}: got {val}, expected {expected}"
    )

# product_exclusion
val, typ = get_val("product_exclusion", "excluded_option")
assert typ == "string", f"product_exclusion: expected string, got {typ}"
assert val == "Cancer_Protect", (
    f"product_exclusion: got {val!r}, expected 'Cancer_Protect'"
)

# =============================================================================
# Sanity: reinstatement proportion should decrease monotonically as period grows
# (longer reinstatement window = more risk = lower proportion of rate difference)
# =============================================================================
prop_ordered = ["30_days", "60_days", "90_days", "180_days", "12_months", "24_months"]
prev_val = None
for key in prop_ordered:
    val, _ = get_val("reinstatement_proportion", key)
    curr = float(val)
    if prev_val is not None:
        assert curr <= prev_val, (
            f"Reinstatement proportion not monotone decreasing at {key}: "
            f"{prev_val} → {curr}"
        )
    prev_val = curr

# Cross-table sanity: reinstatement_period and reinstatement_proportion
# must share the same 6 keys (both cover 30d/60d/90d/180d/12m/24m).
period_keys = set(df.filter(pl.col("feature") == "reinstatement_period")["key"].to_list())
prop_keys   = set(df.filter(pl.col("feature") == "reinstatement_proportion")["key"].to_list())
assert period_keys == prop_keys, (
    f"Key mismatch between period and proportion tables:\n"
    f"  period:     {period_keys}\n"
    f"  proportion: {prop_keys}"
)

print(f"✓ {df.height} rows validated")
print(f"✓ 4 features present")
print(f"✓ Row counts: period=6, proportion=6, discount=4, exclusion=1")
print(f"✓ Reinstatement proportion monotonically decreasing (actuarial invariant)")
print(f"✓ Period and proportion tables share same 6 keys")
print(f"✓ All 17 frozen value checks passed")

print(f"\nType distribution:")
print(df.group_by("param_type").len().sort("param_type"))

print(f"\nFinal DataFrame:")
print(df)

✓ 17 rows validated
✓ 4 features present
✓ Row counts: period=6, proportion=6, discount=4, exclusion=1
✓ Reinstatement proportion monotonically decreasing (actuarial invariant)
✓ Period and proportion tables share same 6 keys
✓ All 17 frozen value checks passed

Type distribution:
shape: (3, 2)
┌────────────┬─────┐
│ param_type ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ float      ┆ 8   │
│ int        ┆ 8   │
│ string     ┆ 1   │
└────────────┴─────┘

Final DataFrame:
shape: (17, 6)
┌──────────────────┬─────────────────┬────────────────┬────────────┬────────────┬──────────────────┐
│ feature          ┆ key             ┆ value          ┆ param_type ┆ source_row ┆ notes            │
│ ---              ┆ ---             ┆ ---            ┆ ---        ┆ ---        ┆ ---              │
│ str              ┆ str             ┆ str            ┆ str        ┆ i64        ┆ str              │
╞══════════════════╪═════════════════╪════════════════╪════════════╪══════════

In [5]:
df.write_parquet(PARQUET_PATH)
parquet_bytes = PARQUET_PATH.read_bytes()
PARQUET_SHA = hashlib.sha256(parquet_bytes).hexdigest()
PARQUET_SIZE_KB = len(parquet_bytes) / 1024

print(f"✓ Wrote {PARQUET_PATH.name}")
print(f"  Size:    {PARQUET_SIZE_KB:.2f} KB")
print(f"  SHA-256: {PARQUET_SHA[:16]}...")

master_wb = openpyxl.load_workbook(MASTER_XLSX)
if SHEET_NAME in master_wb.sheetnames:
    del master_wb[SHEET_NAME]

ws_out = master_wb.create_sheet(SHEET_NAME)
ws_out.append(df.columns)
for row in df.iter_rows():
    ws_out.append(list(row))

master_wb.save(MASTER_XLSX)
print(f"✓ Master workbook updated: {MASTER_XLSX.name}")

✓ Wrote ci_reinstatement.parquet
  Size:    3.40 KB
  SHA-256: f25c3cf9b4f35c59...
✓ Master workbook updated: assumption_master_A2026_07.xlsx


In [6]:
LOG_HEADERS = [
    "built_at", "built_by", "sheet", "rows", "cols",
    "parquet_path", "parquet_size_kb", "parquet_sha256",
    "source_file", "source_sheet", "source_sha256",
    "assumption_version", "notes",
]

log_row = {
    "built_at": BUILT_AT,
    "built_by": BUILT_BY,
    "sheet": SHEET_NAME,
    "rows": df.height,
    "cols": len(df.columns),
    "parquet_path": str(PARQUET_PATH.relative_to(BASE)),
    "parquet_size_kb": f"{PARQUET_SIZE_KB:.2f}",
    "parquet_sha256": PARQUET_SHA,
    "source_file": SOURCE_XLSX.name,
    "source_sheet": SOURCE_SHEET,
    "source_sha256": SOURCE_SHA,
    "assumption_version": ASSUMPTION_VERSION,
    "notes": (
        "CI reinstatement tables and product exclusion. 4 features in one "
        "long-format sheet: reinstatement_period (6), reinstatement_proportion "
        "(6), discount_reduction_factor (4), product_exclusion (1). "
        "Key vocabulary drift: discount table uses '365_days' where the other "
        "two tables use '12_months' — preserved verbatim per rule 5.4; pricing "
        "engine will need a small normaliser. Actuarial invariant enforced: "
        "reinstatement_proportion monotonically decreasing. CI_Standalone_Loading "
        "and CI_Industry_Loading_New live on CI Basis sheet — they are BASIS, "
        "not assumptions; NOT imported here."
    ),
}

write_header = not BUILD_LOG.exists()
with BUILD_LOG.open("a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=LOG_HEADERS)
    if write_header:
        writer.writeheader()
    writer.writerow(log_row)

print(f"✓ build_log.csv updated")
print(f"\n{'='*60}")
print(f"BUILD COMPLETE: {SHEET_NAME}")
print(f"{'='*60}")

✓ build_log.csv updated

BUILD COMPLETE: ci_reinstatement
